# 🏥 01h — Pipeline établissements de santé (FINESS)

Construit `dim_etablissements_sante.parquet` (dept × année) : nombre
d'hôpitaux / pharmacies / laboratoires par département et `par an`. Lancer
`00_config_commun.ipynb` avant. Code repris et adapté du notebook d'une
collègue (`donnees_geographiques.ipynb`).

Source : FINESS (Fichier National des Établissements Sanitaires et Sociaux),
extraction historique via data.gouv.fr.
https://www.data.gouv.fr/datasets/finess-extraction-du-fichier-des-etablissements
Fichiers : `data/raw/etablissements_sante/donnees_2004_2025/etalab_stock_et_YYYY1231.csv`
(un fichier par fin d'année, 2004-2025 — on ne garde que ANNEE_DEBUT-ANNEE_FIN).

Complète (ne remplace pas) `densite_med_gen`/`densite_spe` de `dim_geo_pop` :
c'est une mesure de l'offre de soins par le nombre de structures, pas par le
nombre de praticiens.

In [ ]:
# Préambule : on se place dans le répertoire racine du projet et on ajoute le répertoire courant au PYTHONPATH pour pouvoir importer src/config.py

# pour recharger automatiquement les modules modifiés （src config surtout） sans redémarrer le kernel
%load_ext autoreload 
%autoreload 2

import os
import sys
from pathlib import Path
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
from src.config import RAW_DIR, TABLES_DIR, ANNEE_DEBUT, ANNEE_FIN, DEPTS,  DEPT_NOM_TO_CODE
from src.validation import valider_dim_table

print(
    f"Config chargée depuis src/config.py : {len(DEPTS)} départements | {ANNEE_DEBUT}–{ANNEE_FIN}")
print(f"RAW_DIR    = {RAW_DIR}")
print(f"TABLES_DIR = {TABLES_DIR}")

In [ ]:
# Catégories FINESS retenues -> regroupement simplifié
STRUCT_TO_LABEL = {
    "Hôpitaux Locaux": "Hopitaux",
    "Centres Hospitaliers": "Hopitaux",
    "Com.Biens Usage Méd.": "Pharmacies",
    "Labo Biolog Médicale": "Laboratoires",
    "Labo. d'Analyses": "Laboratoires",
}


def build_dim_etablissements_sante() -> pd.DataFrame:
    """
    Construit dim_etablissements_sante à partir des extractions FINESS
    annuelles (un fichier par fin d'année).

    CLÉ PRIMAIRE : dept × annee
    COLONNES : Hopitaux, Pharmacies, Laboratoires (nb d'établissements)
    """
    dossier = RAW_DIR / "etablissements_sante" / "donnees_2004_2025"
    if not dossier.exists():
        print(f"⚠️  Dossier manquant : {dossier}")
        return pd.DataFrame()

    dfs = []
    for annee in range(ANNEE_DEBUT, ANNEE_FIN + 1):
        fpath = dossier / f"etalab_stock_et_{annee}1231.csv"
        if not fpath.exists():
            print(f"  ⚠️  Manquant : {fpath.name}")
            continue
        df_annee = pd.read_csv(
            fpath, encoding="latin1", sep=";", on_bad_lines="skip",
            dtype={"departement": str},
            usecols=["rslongue", "libdepartement",
                     "departement", "categretab", "libcategretab"],
        )
        df_annee["annee"] = annee
        dfs.append(df_annee)
        print(f"  Chargé {annee} : {len(df_annee):,} lignes")

    df = pd.concat(dfs, ignore_index=True)
    print(f"\n  Total brut : {len(df):,} lignes")

    # Ne garder que les 3 types de structures qui nous intéressent
    df["libcategretab"] = df["libcategretab"].str.strip()
    df = df[df["libcategretab"].isin(STRUCT_TO_LABEL)].copy()
    df["type_structure"] = df["libcategretab"].map(STRUCT_TO_LABEL)

    df["dept"] = df["departement"].astype(str).str.strip().str.upper().str.zfill(2) #str.strip() suppri;er les espaces avant/après / zfill(2) pour les départements à 1 chiffre (ex. 2A → 2A, 1 → 01)
    avant = len(df)
    df = df[df["dept"].isin(DEPTS)]
    print(f"  Après filtre dept + types de structures : {avant:,} → {len(df):,} lignes")

    # Compte par dept x annee x type, puis pivot une colonne par type
    # Résumé nombre par type de structure par département et par année, il faudra le mettre en perpective du nombre d'habitants
    df_agg = df.groupby(["dept", "annee", "type_structure"]).size().reset_index(name="nb")
    
    # pivot et mise au format du département
    df_wide = df_agg.pivot_table(
        index=["dept", "annee"], columns="type_structure", values="nb"
    ).reset_index().fillna(0)
    df_wide.columns.name = None
    for col in ["Hopitaux", "Pharmacies", "Laboratoires"]:
        if col in df_wide.columns:
            df_wide[col] = df_wide[col].astype(int)

    print(f"\n  ✅ dim_etablissements_sante : {df_wide.shape[0]:,} lignes × {df_wide.shape[1]} colonnes")
    print(f"  Départements : {df_wide['dept'].nunique()} | Années : {sorted(df_wide['annee'].unique())}")
    return df_wide


# ══════════════════════════════════════════════════════════════════════════════
# EXÉCUTION
# ══════════════════════════════════════════════════════════════════════════════
dim_etablissements_sante = build_dim_etablissements_sante()

if not dim_etablissements_sante.empty:
    valider_dim_table(dim_etablissements_sante, "dim_etablissements_sante", cle=["dept", "annee"])
    dim_etablissements_sante.to_parquet(TABLES_DIR / "dim_etablissements_sante.parquet", index=False)
    print(f"\n✅ Sauvegardé → data/processed/dim_etablissements_sante.parquet")
    display(dim_etablissements_sante.head(10))

  Chargé 2020 : 95,594 lignes


  Chargé 2021 : 96,906 lignes


  Chargé 2022 : 98,554 lignes


/var/folders/ns/x6_m8h8s6pjg7n9j529g49cc0000gn/T/ipykernel_23178/3729020556.py:30: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  df_annee = pd.read_csv(


  Chargé 2023 : 101,629 lignes


...

  Chargé 2024 : 103,332 lignes


/var/folders/ns/x6_m8h8s6pjg7n9j529g49cc0000gn/T/ipykernel_23178/3729020556.py:30: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  df_annee = pd.read_csv(


  Chargé 2025 : 104,874 lignes

  Total brut : 600,889 lignes
  Après filtre dept + types de structures : 139,653 → 135,415 lignes

  ✅ dim_etablissements_sante : 575 lignes × 5 colonnes
  Départements : 96 | Années : [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
── Validation de dim_etablissements_sante ──


  ✅ Tous les codes dept sont valides (96 départements)
  ✅ Aucun doublon sur la clé ['dept', 'annee']
  ✅ OK — prêt pour la fusion (dim_etablissements_sante)


✅ Sauvegardé → data/processed/dim_etablissements_sante.parquet


,dept,annee,Hopitaux,Laboratoires,Pharmacies
0,01,2020,15,28,166
1,01,2021,15,28,168
2,01,2022,15,28,169
3,01,2023,15,28,167
4,01,2024,15,28,166
5,01,2025,12,0,0
6,02,2020,15,20,182
7,02,2021,15,20,180
8,02,2022,15,21,176
9,02,2023,15,21,171
